In [1]:
import sys
from pathlib import Path

repo_root = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN')
sys.path.insert(0, str(repo_root))


### 1. Configuração do Ambiente

In [ ]:
#!pip install "cognite-sdk[pandas]" matplotlib seaborn tensorflow plotly -q

import os
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import display
from sklearn.preprocessing import RobustScaler
from statsmodels.tsa.stattools import acf
from industrial_ts.dataloader import DataLoader
import json
from getpass import getpass
#import tensorflow as tf
#from tensorflow import keras

print("Bibliotecas importadas com sucesso!")






Updated: UnsupervisedStateSegmenter now supports multi-channel (multivariate) series.

Bibliotecas importadas com sucesso!


/home/ferna/fe/lib/python3.12/site-packages/ipykernel/ipkernel.py:772: UserWarning: You are using version='7.83.1' of the SDK, however version='7.91.2' is available. To suppress this warning, either upgrade or do the following:
>>> from cognite.client.config import global_config
>>> global_config.disable_pypi_version_check = True
  _threading_Thread_run(self)


In [3]:
import debugpy
debugpy.listen(('0.0.0.0', 5678))






('0.0.0.0', 5678)

In [ ]:
import debugpy
debugpy.listen(("0.0.0.0", 5678))
print("aguardando debugger...")
debugpy.wait_for_client()
print("conectado!")


In [4]:
os.environ['COGNITE_CLIENT_SECRET'] = getpass("Enter COGNITE_CLIENT")






### 2. Ativar o DataLoader

In [5]:
import importlib
import sys
importlib.reload(sys.modules['industrial_ts.dataloader'])
from industrial_ts.dataloader import DataLoader
dl = DataLoader()
dl.add_segments(segments=3, window=10, step=10, series=[
    'PH (CBM) 1st Stg ActCompr Poly Head',
    'PH (CBM) 1st Stage ActShaft Power',
    'PH (CBM) 1st Stg ActCompr Poly Head',
    'PH (CBM) 1st Stage ActPress Ratio'  
], path="segmenter_model_10.pkl"
)
#dl.segmenter.save("segmenter_model_10.pkl")








Buscando dados para as 12 séries temporais encontradas.


### 10 Pós-processamento

In [6]:
for i in [2]:
    dl.df['states'].replace(i, 1, inplace=True) # Merge states 1 and 2







/tmp/ipykernel_300436/4230046523.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dl.df['states'].replace(i, 1, inplace=True) # Merge states 1 and 2


In [7]:
dl.add_time_to_change_state_timestamp()






## Baseline TSDF_GRU

In [ ]:
from pathlib import Path
from industrial_ts.gru import TSDF_GRU



In [ ]:
CLAMP_FORWARD = True  # ablation: clamp inside forward?
MAX_DROP = 0.3  # ablation: masking strength

from pathlib import Path
import json
import time
from industrial_ts.gru import TSDF_GRU

# --- Resumo para salvar no JSON ---
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }

# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')


# --- Nome do arquivo baseado na configuracao ---
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_gru(cfg):
    tp = cfg.get('train_params', {})
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    h = cfg.get('hidden_dim', 'H?')
    return _safe(f"tsdf_gru_H{h}_{clamp}_{opt}")
# --- Config base (se nao existir do TSDF_seqKAN) ---
if 'run_config' not in globals():
    batch_size = 512
    run_config = dict(
        model='TSDF_seqKAN',
        in_channels=11,
        hidden_dim=11*32,
        cost_columns=[
            'PH (CBM) 1st Stg ActCompr Poly Head',
            'PH (CBM) 1st Stage ActShaft Power',
            'PH (CBM) 1st Stg ActCompr Poly Head',
            'PH (CBM) 1st Stage ActPress Ratio'
        ],
        lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        sigma_temp=0.6,
        log_likelihood=False,
        use_layernorm=False,
        direct_x=True,
        train_params=dict(
            batch_size=batch_size,
            window_size=10,
            window_step=10,
            epochs=150,
            validate=True,
            patience=20,
            kl_warmup_epochs=20,
            kl_start=0.01,
            rebuild=True,
            reconstruction_test=False,
            warmup_steps=0,
            min_lr_factor=1.0,
            optimizer_name='radam',
            optimizer_params={'lr': 2e-4},
            max_drop=MAX_DROP,  # ablation for masking strength
            x_min=-3.0,
            x_max=3.0,
        ),
    )

# --- defaults se run_config veio de outra celula ---
if 'cost_columns' not in run_config:
    run_config['cost_columns'] = [
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ]
if 'lam' not in run_config:
    run_config['lam'] = [1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
if 'sigma_temp' not in run_config:
    run_config['sigma_temp'] = 0.6
if 'log_likelihood' not in run_config:
    run_config['log_likelihood'] = False
if 'use_layernorm' not in run_config:
    run_config['use_layernorm'] = False
if 'train_params' not in run_config:
    run_config['train_params'] = {}

run_config_gru = dict(
    model='TSDF_GRU',
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    train_params=run_config['train_params'],
)

run_name_gru = _make_run_name_gru(run_config_gru)
out_json_gru = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"TSDF_GRU_final_rebuild_{run_name_gru}.json")

# Garante sem clamp para GRU

model_gru = TSDF_GRU(
    in_channels=run_config_gru['in_channels'],
    hidden_dim=run_config_gru['hidden_dim'],
    cost_columns=run_config_gru['cost_columns'],
    lam=run_config_gru['lam'],
    sigma_temp=run_config_gru['sigma_temp'],
    log_likelihood=run_config_gru['log_likelihood'],
    use_layernorm=run_config_gru['use_layernorm'],
    x_min=run_config_gru['train_params'].get('x_min'),
    x_max=run_config_gru['train_params'].get('x_max'),
    clamp_in_forward=CLAMP_FORWARD,
)

train_start_gru = time.time()
res_gru = model_gru.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config_gru['train_params']['batch_size'],
    window_size=run_config_gru['train_params']['window_size'],
    window_step=run_config_gru['train_params']['window_step'],
    epochs=run_config_gru['train_params']['epochs'],
    validate=run_config_gru['train_params']['validate'],
    patience=run_config_gru['train_params']['patience'],
    kl_warmup_epochs=run_config_gru['train_params']['kl_warmup_epochs'],
    kl_start=run_config_gru['train_params']['kl_start'],
    rebuild=run_config_gru['train_params']['rebuild'],
    reconstruction_test=run_config_gru['train_params']['reconstruction_test'],
    warmup_steps=run_config_gru['train_params']['warmup_steps'],
    min_lr_factor=run_config_gru['train_params']['min_lr_factor'],
    optimizer_name=run_config_gru['train_params']['optimizer_name'],
    optimizer_params=run_config_gru['train_params']['optimizer_params'],
    max_drop=run_config_gru['train_params'].get('max_drop', MAX_DROP),
)
train_end_gru = time.time()
train_seconds_gru = train_end_gru - train_start_gru

res_gru = [r for r in res_gru if r is not None]
out_dir = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results')
out_dir.mkdir(parents=True, exist_ok=True)
gru_ckpt = out_dir / f"TSDF_GRU_{run_name_gru}.pt"
model_gru.save(str(gru_ckpt))
print('saved model:', gru_ckpt)

# Best epoch by test micro
best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res_gru)
summary = _extract_summary(res_gru)

with open(out_json_gru, 'w') as f:
    payload = {
        'config': run_config_gru,
        'results': res_gru,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds_gru,
        'best_epoch': summary['best_epoch'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)

print(res_gru[-1] if res_gru else 'no results')
print('saved:', out_json_gru)


# --- Clamp only at inference/plot (no grad) ---
def clamp_output(x_hat, x_min=None, x_max=None):
    if x_min is not None:
        x_hat = x_hat.clamp(min=x_min)
    if x_max is not None:
        x_hat = x_hat.clamp(max=x_max)
    return x_hat



/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:683: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


GRUPOS (total): {0: 24346, 1: 2637}
GRUPOS (train): {0: 14608, 1: 1582}
GRUPOS (valid): {0: 4869, 1: 528}
GRUPOS (test):  {0: 4869, 1: 527}
[ep 1] batch stats
x_input_raw: min -133.1278 max 2022.7544 p1/p50/p99 -34.8568 0.0162 2.3506 |x|>2 7.90%
x_model_in: min -3.0000 max 3.0000 p1/p50/p99 -2.9664 0.0000 2.0899 |x|>2 2.98%
x_masked: min -21.1221 max 2022.7544 p1/p50/p99 -2.9664 0.0000 2.0899 |x|>2 2.98%
x_hat: min -0.1198 max 0.2024 p1/p50/p99 -0.0835 0.0394 0.1329 |x|>2 0.00%
cc: min 0.0000 max 1.0000 mean 0.2727 nz% 27.27
mask_err: mean 0.339240 nz% 33.92
max_drop (config): None
m_train mean: 0.614808
valid_frac: 0.093058
nobs_bt.mean (raw): 10.236328
nobs_bt min/max (raw): 0.000000 / 24.000000
shapes: x_model_in (512, 10, 11) | x_hat (512, 10, 11) | mask_err (512, 10, 11) | cc (512, 10, 11) | mask_err*cc (512, 10, 11)
x_target: min -3.0000 max 3.0000 p1/p50/p99 -3.0000 0.0162 2.3506 |x|>2 7.90%
x_hat: min -0.1198 max 0.2024 p1/p50/p99 -0.0835 0.0394 0.1329 |x|>2 0.00%
mse_manual(x_

# TSDF_GRU com hidden=24

In [8]:
from pathlib import Path
from industrial_ts.gru import TSDF_GRU



In [9]:
CLAMP_FORWARD = True  # ablation: clamp inside forward?
MAX_DROP = 0.3  # ablation: masking strength

from pathlib import Path
import json
import time
from industrial_ts.gru import TSDF_GRU

# --- Resumo para salvar no JSON ---
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }

# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')


# --- Nome do arquivo baseado na configuracao ---
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_gru(cfg):
    tp = cfg.get('train_params', {})
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    h = cfg.get('hidden_dim', 'H?')
    return _safe(f"tsdf_gru_H{h}_{clamp}_{opt}")
# --- Config base (se nao existir do TSDF_seqKAN) ---
if 'run_config' not in globals():
    batch_size = 512
    run_config = dict(
        model='TSDF_seqKAN',
        in_channels=11,
        hidden_dim=24,
        cost_columns=[
            'PH (CBM) 1st Stg ActCompr Poly Head',
            'PH (CBM) 1st Stage ActShaft Power',
            'PH (CBM) 1st Stg ActCompr Poly Head',
            'PH (CBM) 1st Stage ActPress Ratio'
        ],
        lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        sigma_temp=0.6,
        log_likelihood=False,
        use_layernorm=False,
        direct_x=True,
        train_params=dict(
            batch_size=batch_size,
            window_size=10,
            window_step=10,
            epochs=150,
            validate=True,
            patience=20,
            kl_warmup_epochs=20,
            kl_start=0.01,
            rebuild=True,
            reconstruction_test=False,
            warmup_steps=0,
            min_lr_factor=1.0,
            optimizer_name='radam',
            optimizer_params={'lr': 2e-4},
            max_drop=MAX_DROP,  # ablation for masking strength
            x_min=-3.0,
            x_max=3.0,
        ),
    )

# --- defaults se run_config veio de outra celula ---
if 'cost_columns' not in run_config:
    run_config['cost_columns'] = [
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ]
if 'lam' not in run_config:
    run_config['lam'] = [1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
if 'sigma_temp' not in run_config:
    run_config['sigma_temp'] = 0.6
if 'log_likelihood' not in run_config:
    run_config['log_likelihood'] = False
if 'use_layernorm' not in run_config:
    run_config['use_layernorm'] = False
if 'train_params' not in run_config:
    run_config['train_params'] = {}

run_config_gru = dict(
    model='TSDF_GRU',
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    train_params=run_config['train_params'],
)

run_name_gru = _make_run_name_gru(run_config_gru)
out_json_gru = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"TSDF_GRU_final_rebuild_{run_name_gru}.json")

# Garante sem clamp para GRU

model_gru = TSDF_GRU(
    in_channels=run_config_gru['in_channels'],
    hidden_dim=run_config_gru['hidden_dim'],
    cost_columns=run_config_gru['cost_columns'],
    lam=run_config_gru['lam'],
    sigma_temp=run_config_gru['sigma_temp'],
    log_likelihood=run_config_gru['log_likelihood'],
    use_layernorm=run_config_gru['use_layernorm'],
    x_min=run_config_gru['train_params'].get('x_min'),
    x_max=run_config_gru['train_params'].get('x_max'),
    clamp_in_forward=CLAMP_FORWARD,
)

train_start_gru = time.time()
res_gru = model_gru.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config_gru['train_params']['batch_size'],
    window_size=run_config_gru['train_params']['window_size'],
    window_step=run_config_gru['train_params']['window_step'],
    epochs=run_config_gru['train_params']['epochs'],
    validate=run_config_gru['train_params']['validate'],
    patience=run_config_gru['train_params']['patience'],
    kl_warmup_epochs=run_config_gru['train_params']['kl_warmup_epochs'],
    kl_start=run_config_gru['train_params']['kl_start'],
    rebuild=run_config_gru['train_params']['rebuild'],
    reconstruction_test=run_config_gru['train_params']['reconstruction_test'],
    warmup_steps=run_config_gru['train_params']['warmup_steps'],
    min_lr_factor=run_config_gru['train_params']['min_lr_factor'],
    optimizer_name=run_config_gru['train_params']['optimizer_name'],
    optimizer_params=run_config_gru['train_params']['optimizer_params'],
    max_drop=run_config_gru['train_params'].get('max_drop', MAX_DROP),
)
train_end_gru = time.time()
train_seconds_gru = train_end_gru - train_start_gru

res_gru = [r for r in res_gru if r is not None]
out_dir = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results')
out_dir.mkdir(parents=True, exist_ok=True)
gru_ckpt = out_dir / f"TSDF_GRU_{run_name_gru}.pt"
model_gru.save(str(gru_ckpt))
print('saved model:', gru_ckpt)

# Best epoch by test micro
best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res_gru)
summary = _extract_summary(res_gru)

with open(out_json_gru, 'w') as f:
    payload = {
        'config': run_config_gru,
        'results': res_gru,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds_gru,
        'best_epoch': summary['best_epoch'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)

print(res_gru[-1] if res_gru else 'no results')
print('saved:', out_json_gru)


# --- Clamp only at inference/plot (no grad) ---
def clamp_output(x_hat, x_min=None, x_max=None):
    if x_min is not None:
        x_hat = x_hat.clamp(min=x_min)
    if x_max is not None:
        x_hat = x_hat.clamp(max=x_max)
    return x_hat



/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


RUN CONFIG: {'model': 'TSDF_GRU', 'grid': None, 'grid_eps': None, 'in_channels': 11, 'hidden_dim': None, 'x_min': -3.0, 'x_max': 3.0}


/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:700: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


>% |x| > 2 : 7.41%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-3.00000000 0.02589729 2.39516473]
clipped: 2386 / 56320 (4.24%)
clipped% per feature:
PH (CBM) 1st Stage ExpPress Ratio: 3.44%
PH (CBM) 1st Stage Poly Head Dev: 1.04%
PH (CBM) 1st Stage ActShaft Power: 4.65%
PH (CBM) 1st Stage Press Rat Dev: 0.59%
PH (CBM) 1st Stg ActCompr Poly Head: 3.77%
PH (CBM) 1st Stage Shft Pwr Dev: 0.33%
PH (CBM) 1st Stage ActCompr Poly Eff: 3.28%
PH (CBM) 1st Stage ExpCompr Poly Eff: 17.93%
PH (CBM) 1st Stage ExpShaft Power: 3.98%
PH (CBM) 1st Stage ActPress Ratio: 3.91%
PH (CBM) 1st Stg ExpCompr Poly Head: 3.69%
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 1/150 | Train(sampled) L1:0.820637 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.819290 ± 0.351218 | Val micro:0.538171 ± 0.009903 
          >> Test macro:0.811960 ± 0.344022 | micro:0.536429 ± 0.009338
>% |x| > 2 : 

# TSDF_SeqKANSeq grid 8/hidden 24/k_x 4  K_m 2  k_h 6 k_out 8

In [8]:
import importlib
import industrial_ts.tsdiffusion as tsd
importlib.reload(tsd)

import industrial_ts.seqKAN as sk
importlib.reload(sk)


<module 'industrial_ts.seqKAN' from '/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/seqKAN.py'>

In [ ]:
CLAMP_FORWARD = True  # ablation: clamp inside forward?
MAX_DROP = 0.3  # ablation: masking strength
USE_KANOUT_REBUILD = True  # x_hat direto do kan_out (sem reconstruct e sem MLP decoder)
# Treino via train_cognite com TSDF_seqKANSeq
from industrial_ts.seqKAN import TSDF_seqKANSeq
import time
import json as _json
from pathlib import Path

def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_tsdf_seqkanseq(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    cell = kp.get('cell', kp.get('hidden', {}))
    grid = cell.get('grid', kp.get('grid', 'g?'))
    grid_eps = cell.get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    h = cfg.get('hidden_dim', 'H?')
    suffix = '_kanout' if globals().get('USE_KANOUT_REBUILD', False) else ''
    return _safe(f"tsdf_seqKANseq_H{h}_g{grid}_ge{grid_eps}_{clamp}_{opt}{suffix}")
# config standalone (nao depende de outras celulas)
grid_eps = globals().get('grid_eps', 0.2)
kan_params_seq = {
    'cell': { 'grid_eps': grid_eps, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
    'output': { 'grid_eps': grid_eps, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
    'topk': {
        'enabled': True,
        'kind': 'structural',
        'conn_mode': 'per_out',
        'score': 'coef_l2',
        'k_x': 4,
        'k_h': 6,
        'k_m': 2,
        'mask_size': 11,
        'k_out': 8,
        'warmup_epochs': 30,
    },
}

batch_size = 512
train_fraction = 0.6

run_config_tsdf_seqkanseq = dict(
    model='TSDF_seqKANSeq',
    in_channels=11,
    hidden_dim=24,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio',
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params_seq,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        optimizer_name='radam',
        optimizer_params={'lr': 2e-4},
        max_drop=MAX_DROP,  # ablation for masking strength
        x_min=-3.0,
        x_max=3.0,
    ),
)

run_name_tsdf_seqkanseq = _make_run_name_tsdf_seqkanseq(run_config_tsdf_seqkanseq)
out_json_tsdf_seqkanseq = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"TSDF_seqKANSeq_final_reconstruct_{run_name_tsdf_seqkanseq}.json")

# --- Resumo para salvar no JSON (mesmo padrao do GRU) ---
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }

def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

model_tsdf_seqkanseq = TSDF_seqKANSeq(
    in_channels=run_config_tsdf_seqkanseq['in_channels'],
    hidden_dim=run_config_tsdf_seqkanseq['hidden_dim'],
    cost_columns=run_config_tsdf_seqkanseq['cost_columns'],
    lam=run_config_tsdf_seqkanseq['lam'],
    sigma_temp=run_config_tsdf_seqkanseq['sigma_temp'],
    log_likelihood=run_config_tsdf_seqkanseq['log_likelihood'],
    use_layernorm=run_config_tsdf_seqkanseq['use_layernorm'],
    direct_x=run_config_tsdf_seqkanseq['direct_x'],
    kan_params=run_config_tsdf_seqkanseq['kan_params'],
    x_min=run_config_tsdf_seqkanseq['train_params'].get('x_min'),
    x_max=run_config_tsdf_seqkanseq['train_params'].get('x_max'),
    noise_on_mask=False,
    clamp_in_forward=CLAMP_FORWARD,
)
import torch
device = torch.device('cuda')
model_tsdf_seqkanseq = model_tsdf_seqkanseq.to(device)

train_start_tsdf_seqkanseq = time.time()
res_tsdf_seqkanseq = model_tsdf_seqkanseq.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config_tsdf_seqkanseq['train_params']['batch_size'],
    window_size=run_config_tsdf_seqkanseq['train_params']['window_size'],
    window_step=run_config_tsdf_seqkanseq['train_params']['window_step'],
    epochs=run_config_tsdf_seqkanseq['train_params']['epochs'],
    validate=run_config_tsdf_seqkanseq['train_params']['validate'],
    patience=run_config_tsdf_seqkanseq['train_params']['patience'],
    kl_warmup_epochs=run_config_tsdf_seqkanseq['train_params']['kl_warmup_epochs'],
    kl_start=run_config_tsdf_seqkanseq['train_params']['kl_start'],
    rebuild=run_config_tsdf_seqkanseq['train_params']['rebuild'],
    reconstruction_test=run_config_tsdf_seqkanseq['train_params']['reconstruction_test'],
    warmup_steps=run_config_tsdf_seqkanseq['train_params']['warmup_steps'],
    min_lr_factor=run_config_tsdf_seqkanseq['train_params']['min_lr_factor'],
    optimizer_name=run_config_tsdf_seqkanseq['train_params']['optimizer_name'],
    optimizer_params=run_config_tsdf_seqkanseq['train_params']['optimizer_params'],
    max_drop=run_config_tsdf_seqkanseq['train_params'].get('max_drop', MAX_DROP),
    device=device,
)
train_end_tsdf_seqkanseq = time.time()
train_seconds_tsdf_seqkanseq = train_end_tsdf_seqkanseq - train_start_tsdf_seqkanseq

res_tsdf_seqkanseq = [r for r in res_tsdf_seqkanseq if r is not None]

out_dir = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results')
out_dir.mkdir(parents=True, exist_ok=True)
tsdf_seqkanseq_ckpt = out_dir / f"TSDF_seqKANSeq_{run_name_tsdf_seqkanseq}.pt"
import torch
device = torch.device('cuda')
torch.save(model_tsdf_seqkanseq.state_dict(), tsdf_seqkanseq_ckpt)
print('saved model:', tsdf_seqkanseq_ckpt)

# resumo no mesmo padrao do GRU
best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res_tsdf_seqkanseq)
summary = _extract_summary(res_tsdf_seqkanseq)

with open(out_json_tsdf_seqkanseq, 'w') as f:
    payload = {
        'config': run_config_tsdf_seqkanseq,
        'results': res_tsdf_seqkanseq,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds_tsdf_seqkanseq,
        'best_epoch': summary['best_epoch'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    _json.dump(payload, f)

print(res_tsdf_seqkanseq[-1] if res_tsdf_seqkanseq else 'no results')
print('saved:', out_json_tsdf_seqkanseq)


/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


RUN CONFIG: {'model': 'TSDF_seqKANSeq', 'grid': 8, 'grid_eps': 0.2, 'in_channels': 11, 'hidden_dim': 24, 'x_min': -3.0, 'x_max': 3.0}


/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:701: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


CLAMP COUNTS train | PH (CBM) 1st Stage ExpCompr Poly Eff | = -3.0: 21905/161900 (13.53%) | = 3.0: 0/161900 (0.00%)
CLAMP COUNTS valid | PH (CBM) 1st Stage ExpCompr Poly Eff | = -3.0: 7228/53970 (13.39%) | = 3.0: 0/53970 (0.00%)
CLAMP COUNTS test | PH (CBM) 1st Stage ExpCompr Poly Eff | = -3.0: 7044/53960 (13.05%) | = 3.0: 0/53960 (0.00%)
>% |x| > 2 : 6.81%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-3.00000000 0.01382128 2.30909657]
clipped: 2393 / 56320 (4.25%)
clipped% per feature:
PH (CBM) 1st Stage ExpPress Ratio: 3.95%
PH (CBM) 1st Stage Poly Head Dev: 1.04%
PH (CBM) 1st Stage ActShaft Power: 4.63%
PH (CBM) 1st Stage Press Rat Dev: 0.45%
PH (CBM) 1st Stg ActCompr Poly Head: 3.85%
PH (CBM) 1st Stage Shft Pwr Dev: 0.27%
PH (CBM) 1st Stage ActCompr Poly Eff: 3.59%
PH (CBM) 1st Stage ExpCompr Poly Eff: 16.15%
PH (CBM) 1st Stage ExpShaft Power: 4.55%
PH (CBM) 1st Stage ActPress Ratio: 3.96%
PH (CBM) 1st Stg ExpCompr Poly Head: 4.30%
worst_min_feature: PH (CB

# TSDF_seqKANseq grid 8 hidden 64 k_x 6 k_h:  15 / k_out:  20

In [12]:
import importlib
import industrial_ts.tsdiffusion as tsd
importlib.reload(tsd)

import industrial_ts.seqKAN as sk
importlib.reload(sk)




<module 'industrial_ts.seqKAN' from '/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/seqKAN.py'>

In [ ]:
CLAMP_FORWARD = True  # ablation: clamp inside forward?
MAX_DROP = 0.3  # ablation: masking strength
USE_KANOUT_REBUILD = True  # x_hat direto do kan_out (sem reconstruct e sem MLP decoder)
# Treino via train_cognite com TSDF_seqKANSeq
from industrial_ts.seqKAN import TSDF_seqKANSeq
import time
import json as _json
from pathlib import Path

def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_tsdf_seqkanseq(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    cell = kp.get('cell', kp.get('hidden', {}))
    grid = cell.get('grid', kp.get('grid', 'g?'))
    grid_eps = cell.get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    h = cfg.get('hidden_dim', 'H?')
    suffix = '_kanout' if globals().get('USE_KANOUT_REBUILD', False) else ''
    return _safe(f"tsdf_seqKANseq_H{h}_g{grid}_ge{grid_eps}_{clamp}_{opt}{suffix}")
# config standalone (nao depende de outras celulas)
grid_eps = globals().get('grid_eps', 0.2)
kan_params_seq = {
    'cell': { 'grid_eps': grid_eps, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
    'output': { 'grid_eps': grid_eps, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
    'topk': {
        'enabled': True,
        'kind': 'structural',
        'conn_mode': 'per_out',
        'score': 'coef_l2',
        'k_x': 4,
        'k_h': 15,
        'k_m': 2,
        'mask_size': 11,
        'k_out': 20,
        'warmup_epochs': 30,
    },
}

batch_size = 512
train_fraction = 0.6

run_config_tsdf_seqkanseq = dict(
    model='TSDF_seqKANSeq',
    in_channels=11,
    hidden_dim=64,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio',
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params_seq,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        optimizer_name='radam',
        optimizer_params={'lr': 2e-4},
        max_drop=MAX_DROP,  # ablation for masking strength
        x_min=-3.0,
        x_max=3.0,
    ),
)

run_name_tsdf_seqkanseq = _make_run_name_tsdf_seqkanseq(run_config_tsdf_seqkanseq)
out_json_tsdf_seqkanseq = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"TSDF_seqKANSeq_final_reconstruct_{run_name_tsdf_seqkanseq}.json")

# --- Resumo para salvar no JSON (mesmo padrao do GRU) ---
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }

def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

model_tsdf_seqkanseq = TSDF_seqKANSeq(
    in_channels=run_config_tsdf_seqkanseq['in_channels'],
    hidden_dim=run_config_tsdf_seqkanseq['hidden_dim'],
    cost_columns=run_config_tsdf_seqkanseq['cost_columns'],
    lam=run_config_tsdf_seqkanseq['lam'],
    sigma_temp=run_config_tsdf_seqkanseq['sigma_temp'],
    log_likelihood=run_config_tsdf_seqkanseq['log_likelihood'],
    use_layernorm=run_config_tsdf_seqkanseq['use_layernorm'],
    direct_x=run_config_tsdf_seqkanseq['direct_x'],
    kan_params=run_config_tsdf_seqkanseq['kan_params'],
    x_min=run_config_tsdf_seqkanseq['train_params'].get('x_min'),
    x_max=run_config_tsdf_seqkanseq['train_params'].get('x_max'),
    noise_on_mask=False,
    clamp_in_forward=CLAMP_FORWARD,
)
import torch
device = torch.device('cuda')
model_tsdf_seqkanseq = model_tsdf_seqkanseq.to(device)

train_start_tsdf_seqkanseq = time.time()
res_tsdf_seqkanseq = model_tsdf_seqkanseq.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config_tsdf_seqkanseq['train_params']['batch_size'],
    window_size=run_config_tsdf_seqkanseq['train_params']['window_size'],
    window_step=run_config_tsdf_seqkanseq['train_params']['window_step'],
    epochs=run_config_tsdf_seqkanseq['train_params']['epochs'],
    validate=run_config_tsdf_seqkanseq['train_params']['validate'],
    patience=run_config_tsdf_seqkanseq['train_params']['patience'],
    kl_warmup_epochs=run_config_tsdf_seqkanseq['train_params']['kl_warmup_epochs'],
    kl_start=run_config_tsdf_seqkanseq['train_params']['kl_start'],
    rebuild=run_config_tsdf_seqkanseq['train_params']['rebuild'],
    reconstruction_test=run_config_tsdf_seqkanseq['train_params']['reconstruction_test'],
    warmup_steps=run_config_tsdf_seqkanseq['train_params']['warmup_steps'],
    min_lr_factor=run_config_tsdf_seqkanseq['train_params']['min_lr_factor'],
    optimizer_name=run_config_tsdf_seqkanseq['train_params']['optimizer_name'],
    optimizer_params=run_config_tsdf_seqkanseq['train_params']['optimizer_params'],
    max_drop=run_config_tsdf_seqkanseq['train_params'].get('max_drop', MAX_DROP),
    device=device,
)
train_end_tsdf_seqkanseq = time.time()
train_seconds_tsdf_seqkanseq = train_end_tsdf_seqkanseq - train_start_tsdf_seqkanseq

res_tsdf_seqkanseq = [r for r in res_tsdf_seqkanseq if r is not None]

out_dir = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results')
out_dir.mkdir(parents=True, exist_ok=True)
tsdf_seqkanseq_ckpt = out_dir / f"TSDF_seqKANSeq_{run_name_tsdf_seqkanseq}.pt"
import torch
device = torch.device('cuda')
torch.save(model_tsdf_seqkanseq.state_dict(), tsdf_seqkanseq_ckpt)
print('saved model:', tsdf_seqkanseq_ckpt)

# resumo no mesmo padrao do GRU
best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res_tsdf_seqkanseq)
summary = _extract_summary(res_tsdf_seqkanseq)

with open(out_json_tsdf_seqkanseq, 'w') as f:
    payload = {
        'config': run_config_tsdf_seqkanseq,
        'results': res_tsdf_seqkanseq,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds_tsdf_seqkanseq,
        'best_epoch': summary['best_epoch'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    _json.dump(payload, f)

print(res_tsdf_seqkanseq[-1] if res_tsdf_seqkanseq else 'no results')
print('saved:', out_json_tsdf_seqkanseq)

# TSDF seqKANSeq  

In [8]:
import importlib
import industrial_ts.tsdiffusion as tsd
importlib.reload(tsd)

import industrial_ts.seqKAN as sk
importlib.reload(sk)


<module 'industrial_ts.seqKAN' from '/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/seqKAN.py'>

In [ ]:
CLAMP_FORWARD = True  # ablation: clamp inside forward?
MAX_DROP = 0.3  # ablation: masking strength
USE_KANOUT_REBUILD = True  # x_hat direto do kan_out (sem reconstruct e sem MLP decoder)
# Treino via train_cognite com TSDF_seqKANSeq
from industrial_ts.seqKAN import TSDF_seqKANSeq
import time
import json as _json
from pathlib import Path

def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_tsdf_seqkanseq(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    cell = kp.get('cell', kp.get('hidden', {}))
    grid = cell.get('grid', kp.get('grid', 'g?'))
    grid_eps = cell.get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    h = cfg.get('hidden_dim', 'H?')
    suffix = '_kanout' if globals().get('USE_KANOUT_REBUILD', False) else ''
    return _safe(f"tsdf_seqKANseq_H{h}_g{grid}_ge{grid_eps}_{clamp}_{opt}{suffix}")
# config standalone (nao depende de outras celulas)
grid_eps = globals().get('grid_eps', 0.2)
kan_params_seq = {
    'cell': { 'grid_eps': grid_eps, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
    'output': { 'grid_eps': grid_eps, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
    'topk': {
        'enabled': True,
        'kind': 'structural',
        'conn_mode': 'per_out',
        'score': 'coef_l2',
        'k_x': 6,
        'k_h': 15,
        'k_out': 20,
        'warmup_epochs': 30,
    },
}

batch_size = 512
train_fraction = 0.6

run_config_tsdf_seqkanseq = dict(
    model='TSDF_seqKANSeq',
    in_channels=11,
    hidden_dim=64,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio',
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params_seq,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        optimizer_name='radam',
        optimizer_params={'lr': 2e-4},
        max_drop=MAX_DROP,  # ablation for masking strength
        x_min=-3.0,
        x_max=3.0,
    ),
)

run_name_tsdf_seqkanseq = _make_run_name_tsdf_seqkanseq(run_config_tsdf_seqkanseq)
out_json_tsdf_seqkanseq = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"TSDF_seqKANSeq_final_reconstruct_{run_name_tsdf_seqkanseq}.json")

# --- Resumo para salvar no JSON (mesmo padrao do GRU) ---
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }

def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

model_tsdf_seqkanseq = TSDF_seqKANSeq(
    in_channels=run_config_tsdf_seqkanseq['in_channels'],
    hidden_dim=run_config_tsdf_seqkanseq['hidden_dim'],
    cost_columns=run_config_tsdf_seqkanseq['cost_columns'],
    lam=run_config_tsdf_seqkanseq['lam'],
    sigma_temp=run_config_tsdf_seqkanseq['sigma_temp'],
    log_likelihood=run_config_tsdf_seqkanseq['log_likelihood'],
    use_layernorm=run_config_tsdf_seqkanseq['use_layernorm'],
    direct_x=run_config_tsdf_seqkanseq['direct_x'],
    kan_params=run_config_tsdf_seqkanseq['kan_params'],
    x_min=run_config_tsdf_seqkanseq['train_params'].get('x_min'),
    x_max=run_config_tsdf_seqkanseq['train_params'].get('x_max'),
    noise_on_mask=False,
    clamp_in_forward=CLAMP_FORWARD,
)
import torch
device = torch.device('cuda')
model_tsdf_seqkanseq = model_tsdf_seqkanseq.to(device)

train_start_tsdf_seqkanseq = time.time()
res_tsdf_seqkanseq = model_tsdf_seqkanseq.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config_tsdf_seqkanseq['train_params']['batch_size'],
    window_size=run_config_tsdf_seqkanseq['train_params']['window_size'],
    window_step=run_config_tsdf_seqkanseq['train_params']['window_step'],
    epochs=run_config_tsdf_seqkanseq['train_params']['epochs'],
    validate=run_config_tsdf_seqkanseq['train_params']['validate'],
    patience=run_config_tsdf_seqkanseq['train_params']['patience'],
    kl_warmup_epochs=run_config_tsdf_seqkanseq['train_params']['kl_warmup_epochs'],
    kl_start=run_config_tsdf_seqkanseq['train_params']['kl_start'],
    rebuild=run_config_tsdf_seqkanseq['train_params']['rebuild'],
    reconstruction_test=run_config_tsdf_seqkanseq['train_params']['reconstruction_test'],
    warmup_steps=run_config_tsdf_seqkanseq['train_params']['warmup_steps'],
    min_lr_factor=run_config_tsdf_seqkanseq['train_params']['min_lr_factor'],
    optimizer_name=run_config_tsdf_seqkanseq['train_params']['optimizer_name'],
    optimizer_params=run_config_tsdf_seqkanseq['train_params']['optimizer_params'],
    max_drop=run_config_tsdf_seqkanseq['train_params'].get('max_drop', MAX_DROP),
    device=device,
)
train_end_tsdf_seqkanseq = time.time()
train_seconds_tsdf_seqkanseq = train_end_tsdf_seqkanseq - train_start_tsdf_seqkanseq

res_tsdf_seqkanseq = [r for r in res_tsdf_seqkanseq if r is not None]

out_dir = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results')
out_dir.mkdir(parents=True, exist_ok=True)
tsdf_seqkanseq_ckpt = out_dir / f"TSDF_seqKANSeq_{run_name_tsdf_seqkanseq}.pt"
import torch
device = torch.device('cuda')
torch.save(model_tsdf_seqkanseq.state_dict(), tsdf_seqkanseq_ckpt)
print('saved model:', tsdf_seqkanseq_ckpt)

# resumo no mesmo padrao do GRU
best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res_tsdf_seqkanseq)
summary = _extract_summary(res_tsdf_seqkanseq)

with open(out_json_tsdf_seqkanseq, 'w') as f:
    payload = {
        'config': run_config_tsdf_seqkanseq,
        'results': res_tsdf_seqkanseq,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds_tsdf_seqkanseq,
        'best_epoch': summary['best_epoch'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    _json.dump(payload, f)

print(res_tsdf_seqkanseq[-1] if res_tsdf_seqkanseq else 'no results')
print('saved:', out_json_tsdf_seqkanseq)


# TSDF_seqKANseq 

In [ ]:
import importlib
import industrial_ts.tsdiffusion as tsd
importlib.reload(tsd)

import industrial_ts.seqKAN as sk
importlib.reload(sk)




In [ ]:
CLAMP_FORWARD = True  # ablation: clamp inside forward?
MAX_DROP = 0.3  # ablation: masking strength
USE_KANOUT_REBUILD = True  # x_hat direto do kan_out (sem reconstruct e sem MLP decoder)
# Treino via train_cognite com TSDF_seqKANSeq
from industrial_ts.seqKAN import TSDF_seqKANSeq
import time
import json as _json
from pathlib import Path

def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_tsdf_seqkanseq(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    cell = kp.get('cell', kp.get('hidden', {}))
    grid = cell.get('grid', kp.get('grid', 'g?'))
    grid_eps = cell.get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    h = cfg.get('hidden_dim', 'H?')
    suffix = '_kanout' if globals().get('USE_KANOUT_REBUILD', False) else ''
    return _safe(f"tsdf_seqKANseq_H{h}_g{grid}_ge{grid_eps}_{clamp}_{opt}{suffix}")
# config standalone (nao depende de outras celulas)
grid_eps = globals().get('grid_eps', 0.2)
kan_params_seq = {
    'cell': { 'grid_eps': grid_eps, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
    'output': { 'grid_eps': grid_eps, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
    'topk': {
        'enabled': True,
        'kind': 'structural',
        'conn_mode': 'per_out',
        'score': 'coef_l2',
        'k_x': 6,
        'k_h': 15,
        'k_out': 20,
        'warmup_epochs': 30,
    },
}

batch_size = 512
train_fraction = 0.6

run_config_tsdf_seqkanseq = dict(
    model='TSDF_seqKANSeq',
    in_channels=11,
    hidden_dim=64,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio',
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params_seq,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        optimizer_name='radam',
        optimizer_params={'lr': 2e-4},
        max_drop=MAX_DROP,  # ablation for masking strength
        x_min=-3.0,
        x_max=3.0,
    ),
)

run_name_tsdf_seqkanseq = _make_run_name_tsdf_seqkanseq(run_config_tsdf_seqkanseq)
out_json_tsdf_seqkanseq = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"TSDF_seqKANSeq_final_reconstruct_{run_name_tsdf_seqkanseq}.json")

# --- Resumo para salvar no JSON (mesmo padrao do GRU) ---
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }

def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

model_tsdf_seqkanseq = TSDF_seqKANSeq(
    in_channels=run_config_tsdf_seqkanseq['in_channels'],
    hidden_dim=run_config_tsdf_seqkanseq['hidden_dim'],
    cost_columns=run_config_tsdf_seqkanseq['cost_columns'],
    lam=run_config_tsdf_seqkanseq['lam'],
    sigma_temp=run_config_tsdf_seqkanseq['sigma_temp'],
    log_likelihood=run_config_tsdf_seqkanseq['log_likelihood'],
    use_layernorm=run_config_tsdf_seqkanseq['use_layernorm'],
    direct_x=run_config_tsdf_seqkanseq['direct_x'],
    kan_params=run_config_tsdf_seqkanseq['kan_params'],
    x_min=run_config_tsdf_seqkanseq['train_params'].get('x_min'),
    x_max=run_config_tsdf_seqkanseq['train_params'].get('x_max'),
    noise_on_mask=False,
    clamp_in_forward=CLAMP_FORWARD,
)
import torch
device = torch.device('cuda')
model_tsdf_seqkanseq = model_tsdf_seqkanseq.to(device)

train_start_tsdf_seqkanseq = time.time()
res_tsdf_seqkanseq = model_tsdf_seqkanseq.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config_tsdf_seqkanseq['train_params']['batch_size'],
    window_size=run_config_tsdf_seqkanseq['train_params']['window_size'],
    window_step=run_config_tsdf_seqkanseq['train_params']['window_step'],
    epochs=run_config_tsdf_seqkanseq['train_params']['epochs'],
    validate=run_config_tsdf_seqkanseq['train_params']['validate'],
    patience=run_config_tsdf_seqkanseq['train_params']['patience'],
    kl_warmup_epochs=run_config_tsdf_seqkanseq['train_params']['kl_warmup_epochs'],
    kl_start=run_config_tsdf_seqkanseq['train_params']['kl_start'],
    rebuild=run_config_tsdf_seqkanseq['train_params']['rebuild'],
    reconstruction_test=run_config_tsdf_seqkanseq['train_params']['reconstruction_test'],
    warmup_steps=run_config_tsdf_seqkanseq['train_params']['warmup_steps'],
    min_lr_factor=run_config_tsdf_seqkanseq['train_params']['min_lr_factor'],
    optimizer_name=run_config_tsdf_seqkanseq['train_params']['optimizer_name'],
    optimizer_params=run_config_tsdf_seqkanseq['train_params']['optimizer_params'],
    max_drop=run_config_tsdf_seqkanseq['train_params'].get('max_drop', MAX_DROP),
    device=device,
)
train_end_tsdf_seqkanseq = time.time()
train_seconds_tsdf_seqkanseq = train_end_tsdf_seqkanseq - train_start_tsdf_seqkanseq

res_tsdf_seqkanseq = [r for r in res_tsdf_seqkanseq if r is not None]

out_dir = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results')
out_dir.mkdir(parents=True, exist_ok=True)
tsdf_seqkanseq_ckpt = out_dir / f"TSDF_seqKANSeq_{run_name_tsdf_seqkanseq}.pt"
import torch
device = torch.device('cuda')
torch.save(model_tsdf_seqkanseq.state_dict(), tsdf_seqkanseq_ckpt)
print('saved model:', tsdf_seqkanseq_ckpt)

# resumo no mesmo padrao do GRU
best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res_tsdf_seqkanseq)
summary = _extract_summary(res_tsdf_seqkanseq)

with open(out_json_tsdf_seqkanseq, 'w') as f:
    payload = {
        'config': run_config_tsdf_seqkanseq,
        'results': res_tsdf_seqkanseq,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds_tsdf_seqkanseq,
        'best_epoch': summary['best_epoch'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    _json.dump(payload, f)

print(res_tsdf_seqkanseq[-1] if res_tsdf_seqkanseq else 'no results')
print('saved:', out_json_tsdf_seqkanseq)


# Análise de splines (saída via kan_out) – TSDF_seqKANSeq


In [ ]:
# (x_hat via kan_out). 
import torch
import numpy as np
import matplotlib.pyplot as plt

# --- checkpoint mais recente ---
ckpt_path = "/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results/TSDF_seqKANSeq_tsdf_seqKANseq_H24_g8_ge0.2_c-3.0_3.0_radam.pt"

# --- config (usa o já definido se existir, senão recria) ---
if 'run_config_tsdf_seqkanseq' in globals():
    cfg = run_config_tsdf_seqkanseq
else:
    grid_eps = 0.2
    kan_params_seq = {
        'cell': { 'grid_eps': grid_eps, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
        'output': { 'grid_eps': grid_eps, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
        'topk': {
            'enabled': True,
            'kind': 'structural',
            'conn_mode': 'per_out',
            'score': 'coef_l2',
            'k_x': 6,
            'k_h': 6,
        'k_m': 4,
        'mask_size': 11,
            'k_out': 8,
            'warmup_epochs': 30,
        },
    }
    cfg = dict(
        model='TSDF_seqKANSeq',
        in_channels=11,
        hidden_dim=24,
        cost_columns=[
            'PH (CBM) 1st Stg ActCompr Poly Head',
            'PH (CBM) 1st Stage ActShaft Power',
            'PH (CBM) 1st Stg ActCompr Poly Head',
            'PH (CBM) 1st Stage ActPress Ratio',
        ],
        lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        sigma_temp=0.6,
        log_likelihood=False,
        use_layernorm=False,
        direct_x=True,
        kan_params=kan_params_seq,
        train_params=dict(
            batch_size=512,
            window_size=10,
            window_step=10,
            epochs=150,
            validate=True,
            patience=20,
            kl_warmup_epochs=20,
            kl_start=0.01,
            rebuild=True,
            reconstruction_test=False,
            warmup_steps=0,
            min_lr_factor=1.0,
            optimizer_name='radam',
            optimizer_params={'lr': 2e-4},
            max_drop=0.3,
            x_min=-3.0,
            x_max=3.0,
        ),
        train_fraction=0.6,
    )

from industrial_ts.seqKAN import TSDF_seqKANSeq

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TSDF_seqKANSeq(
    in_channels=cfg['in_channels'],
    hidden_dim=cfg['hidden_dim'],
    cost_columns=cfg['cost_columns'],
    lam=cfg['lam'],
    sigma_temp=cfg['sigma_temp'],
    log_likelihood=cfg['log_likelihood'],
    use_layernorm=cfg['use_layernorm'],
    direct_x=cfg['direct_x'],
    kan_params=cfg['kan_params'],
    x_min=cfg['train_params'].get('x_min'),
    x_max=cfg['train_params'].get('x_max'),
    clamp_in_forward=globals().get('CLAMP_FORWARD', True),
).to(device)

state = torch.load(ckpt_path, map_location=device)
missing, unexpected = model.load_state_dict(state, strict=False)
print("load_state_dict missing:", missing)
print("load_state_dict unexpected:", unexpected)
model.eval()

# --- prepara batch (reconstrução) ---
feature_cols = list(dl.df.columns[:-3])
window_size = cfg['train_params']['window_size']
window_step = cfg['train_params']['window_step']

ds = model._make_dataset(
    dl.df,
    timestamp_col='index',
    window_size=window_size,
    feature_cols=feature_cols,
    static_features_cols=None,
    window_step=window_step,
)

x_all = ds.tensors[0]
train_end = max(1, int(len(x_all) * cfg.get('train_fraction', 0.6)))
x_scaled = model.scale_tensor(x_all[:train_end], x_all)

B = min(256, x_scaled.shape[0])
x_batch = x_scaled[:B].to(device)
mask = torch.ones_like(x_batch)

with torch.no_grad():
    out = model(x_batch, return_x_hat=True, mask=mask, test=True)
    x_hat = out[4]

print('x_batch:', tuple(x_batch.shape), 'x_hat:', None if x_hat is None else tuple(x_hat.shape))

# --- captura inputs reais do KAN cell (x vs h) ---
seq = model.encoder_ode_x  # SeqKANSeq usado na reconstrução
inputs_collected = []

def _hook_cell(module, inputs, output):
    inputs_collected.append(inputs[0].detach().cpu())

handle = seq.kan_cell.register_forward_hook(_hook_cell)
with torch.no_grad():
    h_in = torch.cat([x_batch, mask], dim=-1) if model.direct_x else x_batch
    _ = seq(h_in, mask=None, return_last=False)
handle.remove()

if not inputs_collected:
    raise RuntimeError('Nenhum input capturado do KAN cell.')

h_in_samples = torch.cat(inputs_collected, dim=0)
max_samples = 5000
if h_in_samples.shape[0] > max_samples:
    idx = torch.randperm(h_in_samples.shape[0])[:max_samples]
    h_in_samples = h_in_samples[idx]

h_in_samples = h_in_samples.to(device)

# --- ativa save_act e coleta ativação para atributo (KAN cell) ---
kan_cell = seq.kan_cell
old_save_act = getattr(kan_cell, 'save_act', False)
kan_cell.save_act = True
kan_cell.get_act(h_in_samples)

scores = kan_cell.feature_score
if scores is None:
    raise RuntimeError('feature_score não disponível (verifique save_act).')

# scores: (out_dim, in_dim)
if scores.dim() == 1:
    scores_mean = scores
else:
    scores_mean = scores.mean(dim=0)

input_size = seq.input_size
feat_scores_x = scores_mean[:input_size]
feat_scores_h = scores_mean[input_size:]

# nomes das features (x + mask) quando direct_x=True
if model.direct_x:
    x_names = feature_cols + [f"mask:{c}" for c in feature_cols]
else:
    x_names = feature_cols
if len(x_names) != input_size:
    x_names = [f"x{i}" for i in range(input_size)]

# ranking x vs h (soma)
print('Influência total (x vs h):')
print('x:', float(feat_scores_x.sum().item()), 'h:', float(feat_scores_h.sum().item()))

# ranking por feature (x)
print('\nTop features (x):')
_k = min(10, feat_scores_x.numel())
vals, idxs = torch.topk(feat_scores_x, k=_k)
for v, i in zip(vals, idxs):
    print(f"{x_names[int(i)]}: {float(v):.6f}")

# --- plota splines das features mais influentes (KAN cell) ---
print('\nGerando gráficos das splines (KAN cell, top x)...')
plot_k = min(6, feat_scores_x.numel())
vals, idxs = torch.topk(feat_scores_x, k=plot_k)
for i in idxs:
    i = int(i)
    # escolhe o output (neurônio) mais associado a essa feature
    if scores.dim() > 1:
        j = int(torch.argmax(scores[:, i]).item())
    else:
        j = 0
    kan_cell.get_fun(l=0, i=i, j=j)
    plt.title(f"Spline (cell): {x_names[i]} -> out {j}")
    plt.xlabel(x_names[i])
    plt.ylabel('spline(x)')
    plt.show()

# --- separação x vs mask vs h ---
if model.direct_x and len(feature_cols) * 2 == input_size:
    n = len(feature_cols)
    scores_x_only = feat_scores_x[:n]
    scores_mask_only = feat_scores_x[n:]
    print('\nSeparação x vs mask vs h:')
    print('x:', float(scores_x_only.sum().item()), 'mask:', float(scores_mask_only.sum().item()), 'h:', float(feat_scores_h.sum().item()))

# --- KAN out: ranking e splines por head (impacto direto no output) ---
print('\nKAN out: ranking e splines (top features por head)')
head_inputs = {}

def _make_head_hook(i):
    def _hook(module, inputs, output):
        head_inputs.setdefault(i, []).append(inputs[0].detach().cpu())
    return _hook

handles = [head.register_forward_hook(_make_head_hook(i)) for i, head in enumerate(seq.kan_out)]
with torch.no_grad():
    h_in = torch.cat([x_batch, mask], dim=-1) if model.direct_x else x_batch
    _ = seq(h_in, mask=None, return_last=False)
for h in handles:
    h.remove()

for head_idx, head in enumerate(seq.kan_out):
    if head_idx not in head_inputs:
        continue
    h_samples = torch.cat(head_inputs[head_idx], dim=0)
    if h_samples.shape[0] > max_samples:
        idx = torch.randperm(h_samples.shape[0])[:max_samples]
        h_samples = h_samples[idx]
    h_samples = h_samples.to(device)

    old = getattr(head, 'save_act', False)
    head.save_act = True
    head.get_act(h_samples)
    head_scores = head.feature_score
    if head_scores is None:
        head.save_act = old
        continue
    if head_scores.dim() == 2:
        head_scores_mean = head_scores.mean(dim=0)
    else:
        head_scores_mean = head_scores
    k = min(3, head_scores_mean.numel())
    vals, idxs = torch.topk(head_scores_mean, k=k)
    print(f"Head {head_idx}: top {k}")
    for v, i in zip(vals, idxs):
        print(f"  h{i}: {float(v):.6f}")
    # plot spline do head para top1
    i0 = int(idxs[0])
    if head_scores.dim() > 1:
        j0 = int(torch.argmax(head_scores[:, i0]).item())
    else:
        j0 = 0
    head.get_fun(l=0, i=i0, j=j0)
    plt.title(f"Spline (head {head_idx}): h{i0} -> out {j0}")
    plt.xlabel(f"h{i0}")
    plt.ylabel('spline(h)')
    plt.show()
    head.save_act = old

# restaura
kan_cell.save_act = old_save_act
